# Voice access project - cleaned development pipeline

This notebook fixes the main problems from the first version:

- correct meaning of labels: `1 = allow`, `0 = not_allow`
- correct FAR / FRR counting
- split without leakage from the same recording
- speaker-disjoint split for class `0`
- one clean path: recordings -> 3 s segments -> mel spectrogram PNG -> CNN
- place for multiple experiments
- place for adding one more person to class `1`

## 1. Imports and config
Set the paths once.  
If you want, replace `ALLOW_SPEAKERS` with your own manual list.

In [1]:
from pathlib import Path
import pandas as pd
import torch

from voice_project_code import (
    ProjectPaths,
    append_new_allow_speaker,
    build_recording_metadata,
    build_spectrogram_pngs,
    make_loaders,
    predict_wav_file,
    rebuild_after_new_person,
    run_experiment,
    segment_recordings,
    set_seed,
    validate_recording_metadata,
    validate_segment_metadata,
)

SOURCE_DIR = Path(r"VOiCES_devkit/source-16k")
PATHS = ProjectPaths.from_work_dir("voice_project_artifacts")

SEED = 123
ALLOW_SPEAKERS = None
N_ALLOW_SPEAKERS = 5

SEGMENT_SECONDS = 3.0
KEEP_REMAINDER = False
TRIM_SILENCE = False
TOP_DB = 30.0

IMAGE_SIZE = 128
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 128
FMIN = 20
FMAX = 8000

BATCH_SIZE = 32
EPOCHS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

set_seed(SEED)
print("DEVICE =", DEVICE)
print("WORK DIR =", PATHS.work_dir.resolve())

DEVICE = cpu
WORK DIR = C:\Users\idani\PycharmProjects\ComputerVision\voice_project_artifacts


## 2. Build recording-level metadata

Design choice:

- class `1` (`allow`): split **within speaker** by recordings, so the same allowed speaker can appear in train / valid / test, but never the same recording
- class `0` (`not_allow`): split **by speakers**, so negative speakers do not overlap across splits

In [2]:
recording_meta = build_recording_metadata(
    source_dir=SOURCE_DIR,
    paths=PATHS,
    allow_speakers=ALLOW_SPEAKERS,
    n_allow_speakers=N_ALLOW_SPEAKERS,
    seed=SEED,
)

recording_summary = validate_recording_metadata(recording_meta)

print("recording_meta saved to:", PATHS.recording_meta_path)
print(recording_meta.head())
print()
print(pd.Series(recording_summary))

OSError: Cannot save file into a non-existent directory: 'voice_project_artifacts'

## 3. Segment audio into 3-second clips
This is where leakage must still stay impossible: every segment inherits the split from its source recording.

In [ ]:
segment_meta = segment_recordings(
    paths=PATHS,
    segment_seconds=SEGMENT_SECONDS,
    keep_remainder=KEEP_REMAINDER,
    trim_silence=TRIM_SILENCE,
    top_db=TOP_DB,
    clean_output=True,
)

segment_summary = validate_segment_metadata(segment_meta)

print("segment_meta saved to:", PATHS.segment_meta_path)
print(segment_meta.head())
print()
print(pd.Series(segment_summary))

## 4. Convert segments to mel spectrogram PNG files
The saved metadata keeps the original label, so we do not depend on `ImageFolder` and we avoid the old label-order bug.

In [ ]:
spectro_meta = build_spectrogram_pngs(
    paths=PATHS,
    image_size=IMAGE_SIZE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    fmin=FMIN,
    fmax=FMAX,
    clean_output=True,
)

print("spectro_meta saved to:", PATHS.spectro_meta_path)
print(spectro_meta.head())
print()
print(spectro_meta.groupby(["split", "label"]).size())

## 5. Sanity check of loaders and normalization
Mean and std are computed only from train.

In [ ]:
train_loader, valid_loader, test_loader, loader_stats = make_loaders(
    paths=PATHS,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalize=True,
    augment=False,
)

print(pd.Series(loader_stats))

## 6. Baseline experiment
Start with a smaller CNN.  
This is a better baseline than `resnet18` when class `1` is still tiny.

In [ ]:
history_baseline, results_baseline = run_experiment(
    paths=PATHS,
    experiment_name="baseline_smallcnn_adam",
    model_name="smallcnn",
    dropout=0.30,
    use_batchnorm=True,
    optimizer_name="adam",
    lr=1e-3,
    weight_decay=1e-4,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    normalize=True,
    augment=False,
    device=DEVICE,
)

display(history_baseline.tail())
display(results_baseline)

## 7. More experiments
You need multiple comparisons in the project.  
Below are ready configs for at least four different components:

- optimizer: Adam vs SGD
- dropout: 0.30 vs 0.00
- batch normalization: on vs off
- architecture: SmallCNN vs ResNet18-from-scratch
- optional: transfer learning comparison

In [ ]:
EXPERIMENT_CONFIGS = [
    {
        "experiment_name": "exp_smallcnn_adam_bn_dropout",
        "model_name": "smallcnn",
        "dropout": 0.30,
        "use_batchnorm": True,
        "optimizer_name": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    {
        "experiment_name": "exp_smallcnn_sgd_bn_dropout",
        "model_name": "smallcnn",
        "dropout": 0.30,
        "use_batchnorm": True,
        "optimizer_name": "sgd",
        "lr": 1e-2,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    {
        "experiment_name": "exp_smallcnn_adam_no_bn",
        "model_name": "smallcnn",
        "dropout": 0.30,
        "use_batchnorm": False,
        "optimizer_name": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    {
        "experiment_name": "exp_smallcnn_adam_no_dropout",
        "model_name": "smallcnn",
        "dropout": 0.00,
        "use_batchnorm": True,
        "optimizer_name": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    {
        "experiment_name": "exp_resnet18_scratch",
        "model_name": "resnet18_scratch",
        "dropout": 0.30,
        "use_batchnorm": True,
        "optimizer_name": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    # Optional:
    # {
    #     "experiment_name": "exp_resnet18_transfer",
    #     "model_name": "resnet18_transfer",
    #     "dropout": 0.30,
    #     "use_batchnorm": True,
    #     "optimizer_name": "adam",
    #     "lr": 1e-4,
    #     "weight_decay": 1e-4,
    #     "epochs": EPOCHS,
    #     "batch_size": BATCH_SIZE,
    #     "image_size": IMAGE_SIZE,
    #     "normalize": True,
    #     "augment": False,
    #     "device": DEVICE,
    # },
]

In [ ]:
all_results = []

for cfg in EXPERIMENT_CONFIGS:
    _, results_df = run_experiment(paths=PATHS, **cfg)
    all_results.append(results_df)

comparison_df = pd.concat(all_results, ignore_index=True)
comparison_df = comparison_df.sort_values(["valid_far", "valid_frr", "test_far", "test_frr"])

comparison_df

## 8. Add one more person to class 1
When you get a new folder with WAV files for one more allowed speaker:

1. append that speaker to class `1`
2. rebuild segments and spectrograms
3. fine-tune or retrain the selected model

In [ ]:
# Example:
# NEW_PERSON_DIR = Path(r"new_allow_speakers/sp9999")
# updated_recording_meta = append_new_allow_speaker(
#     paths=PATHS,
#     new_person_dir=NEW_PERSON_DIR,
#     seed=SEED,
# )
# updated_segment_meta, updated_spectro_meta = rebuild_after_new_person(
#     paths=PATHS,
#     segment_seconds=SEGMENT_SECONDS,
#     keep_remainder=KEEP_REMAINDER,
#     trim_silence=TRIM_SILENCE,
#     top_db=TOP_DB,
#     image_size=IMAGE_SIZE,
#     n_fft=N_FFT,
#     hop_length=HOP_LENGTH,
#     n_mels=N_MELS,
#     fmin=FMIN,
#     fmax=FMAX,
# )
# 
# history_updated, results_updated = run_experiment(
#     paths=PATHS,
#     experiment_name="baseline_smallcnn_after_new_person",
#     model_name="smallcnn",
#     dropout=0.30,
#     use_batchnorm=True,
#     optimizer_name="adam",
#     lr=5e-4,
#     weight_decay=1e-4,
#     epochs=5,
#     batch_size=BATCH_SIZE,
#     image_size=IMAGE_SIZE,
#     normalize=True,
#     augment=False,
#     device=DEVICE,
# )
# 
# display(results_updated)

## 9. Quick inference on a WAV file
This is the core of the final notebook: load one best model and classify a new recording.

In [ ]:
# Example:
# BEST_CHECKPOINT = PATHS.model_dir / "baseline_smallcnn_adam.pt"
# wav_result = predict_wav_file(
#     wav_path=Path(r"some_test_audio.wav"),
#     checkpoint_path=BEST_CHECKPOINT,
#     model_name="smallcnn",
#     train_mean=loader_stats["train_mean"],
#     train_std=loader_stats["train_std"],
#     segment_seconds=SEGMENT_SECONDS,
#     image_size=IMAGE_SIZE,
#     device=DEVICE,
#     dropout=0.30,
#     use_batchnorm=True,
# )
# 
# wav_result

## 10. What to do next

Recommended order:

1. run the baseline
2. run the comparison table
3. check whether FAR and FRR are both reasonable and not too far apart
4. later add augmentation / noise experiments
5. build the final notebook around the best checkpoint